# Vesuvius First Letters — ink sweep

Runs the ink stage of the winning PHerc0343 submission (Erwin Nieuwlaar, $50K,
9 Oct 2026) on either a **positive control** with known ink or a **target** patch,
through an identical path so the two are comparable.

| | |
|---|---|
| `control` | PHerc0139 w035 — published 28-layer render **plus hand-drawn ink labels**. Tells us the pipeline works and what recovered ink looks like. |
| `pherc0800` | published 31-layer renders of an eligible scroll, no geometry work needed |

Set `WHAT` in the config cell; everything else follows.

### Before running

1. **Settings → Accelerator → GPU T4 ×2**, **Internet → On** (needs phone verification).
2. **Keep this notebook private.** Prize terms: the discovery must not be public until
   the prize is announced, and a public notebook publishes its outputs.
3. Run top to bottom. The sweep is resumable — finished maps are skipped.


## 1. Machine


In [ ]:
!nvidia-smi -L
import subprocess
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
major = int(float(drv.split('.')[0]))
BACKEND = 'cu130' if major >= 580 else ('cu128' if major >= 525 else 'cu126')
print('driver', drv, '-> torch backend', BACKEND)


## 2. Config — the only cell you edit

Depth range: the villa models read a 17-layer window centred on `N/2 + d`, so a
31-layer render allows `d = -7..+7` and the 28-layer control `d = -6..+5`. The scripts
derive this from the render; `DMIN/DMAX` only narrows it. Start narrow — the central
depths are where ink sits if the mesh sits on the sheet — then widen if it comes back
empty.

PHerc0800 segments, by canvas area (four of them are the *same sheet*, so running all
is mostly re-answering one question):

- `20251028222030` 4.97 cm² — biggest; **swept, null at d −3..+3**
- `20251028220955` 3.29 cm² — the one genuinely different wrap, best remaining
- `20251028225813` 3.88 cm², `20251028213516` 3.61 cm², `20251028220042` 3.05 cm² — same sheet as 222030
- `20251029010146` 0.72 cm² — different again, but small


In [ ]:
WHAT = 'pherc0800'              # 'control' | 'pherc0800'
SEG  = '20251028222030'       # used when WHAT == 'pherc0800'
DMIN, DMAX = -7, 7
VILLA_PAR, HEC_PAR = 2, 2     # concurrent jobs per queue; ~3 GB each on a 15 GB T4
MODELS = 'dnative hybrid_3d2d-seed42 hybrid_3d2d-seed43'   # hecate dropped: see below

import os
os.environ['ROOT'] = ROOT = '/kaggle/temp/data'
os.environ['TORCH_BACKEND'] = BACKEND
REPO = '/kaggle/temp/ink-det'
PY   = ROOT + '/venv/bin/python'
NAME = 'w035_control' if WHAT == 'control' else SEG
RENDER = f'{ROOT}/control/{NAME}.zarr' if WHAT == 'control' else f'{ROOT}/renders/{NAME}.zarr'
LABELS = f'{ROOT}/control/{NAME}' if WHAT == 'control' else None
PX_UM  = 9.362 if WHAT == 'control' else 8.64
MAPS, IMAGES = f'{ROOT}/maps/{NAME}', f'{ROOT}/images/{NAME}'
SAVE = f'/kaggle/working/{NAME}'
# Published team renders have their normals the other way up from Erwin's own, so
# villa's forward pass - which ensemble_maps labels _blind - is the READABLE face,
# and it is written rotated 180 deg. Measured on the control: 0.972 that way round,
# 0.588 the other. Both PHerc0139 w035 and PHerc0800 are team renders.
FACE, ROT180 = 'blind', True
print(WHAT, NAME, '->', RENDER)


## 3. Environment

villa pinned to `0fb38c45` (the commit every map of the winning submission was computed
with), CPython 3.14 (`vesuvius` pins `>=3.14,<3.15`), the four public checkpoints with
sha256 verification, and the patch that lets the ensemble step read renders that are not
66 layers deep. Takes ~15 min the first time; re-running is a no-op.


In [ ]:
# fail loudly: no internet means no repo, no S3, no checkpoints, and every
# later cell fails with a confusing 'No such file or directory'
import subprocess, sys
if subprocess.run(['curl', '-sI', '--max-time', '20', 'https://github.com'],
                  capture_output=True).returncode != 0:
    sys.exit('No internet. Turn on Settings -> Internet (needs phone verification).')
!mkdir -p /kaggle/temp
!test -d {REPO}/.git && (cd {REPO} && git pull -q) || git clone https://github.com/Ayyyn/ink-det.git {REPO}
!test -f {REPO}/sprint/setup_gpu.sh || echo 'CLONE FAILED - stop here and read the error above'
!bash {REPO}/sprint/setup_gpu.sh {ROOT} 2>&1 | tail -30


### What the checkpoints expect

villa takes the layer range given by `--layer-start/--layer-end`, centre-crops it to
the checkpoint's own `input_depth`, and if the selection is **shorter** than that, the
loader centre-places it in a zeroed array — silent zero padding. Erwin's script passes
17 layers, which is correct only if every model's `input_depth` is at most 17. This
prints what each checkpoint actually declares.


In [ ]:
!{PY} {REPO}/sprint/checkpoint_info.py {ROOT}/ckpt


## 4. Get the render

The control is cropped to a window over the labelled ink so it is comparable in size to
a PHerc0800 segment (the full w035 render is 26.7 cm², ~5× the cost). `make_control.py`
writes the zarr store by hand and reads it back before returning, so a bad write fails
here rather than halfway through a sweep.


In [ ]:
if WHAT == 'control':
    !{PY} {REPO}/sprint/make_control.py --out {ROOT}/control
else:
    !{PY} {REPO}/sprint/pull_render.py --scroll PHerc0800 --segment {SEG} --out {ROOT}/renders
!{PY} {REPO}/sprint/render_info.py {RENDER}


## 5. Look before modelling

Ink is sometimes visible directly in a flattened render, and that alone qualifies.

The depth profile matters as much as the picture: it shows where the sheet actually sits
in the render. If the density peak is far from the centre the mesh is off the sheet and
the swept depths are aimed at nothing. On PHerc0800 the peak was layer 12–14 of 31, so
d = −3..+3 was correctly aimed — which is what made that null trustworthy.


In [ ]:
# this cell runs in Kaggle's own kernel, not the venv the sweep uses, and the base
# image does not always carry zarr - install it rather than failing three cells in
try:
    import zarr
except ImportError:
    !pip install -q zarr
    import zarr
import numpy as np
import matplotlib.pyplot as plt
a = zarr.open(RENDER, mode='r')['0']
st = np.asarray(a[:])
nz = st.shape[0]
valid = (st > 0).all(0)
prof = np.array([st[z][valid].mean() for z in range(nz)])
peak = int(prof.argmax())
fig, ax = plt.subplots(1, 2, figsize=(16, 7),
                       gridspec_kw={'width_ratios': [3, 1]})
mid = st[nz // 2]
m = mid > 0
lo, hi = np.percentile(mid[m].astype(np.float32), [1, 99.5])
ax[0].imshow(np.clip((mid - lo) / (hi - lo), 0, 1) * m, cmap='gray')
ax[0].set_title(f'{NAME} centre layer {nz // 2}'); ax[0].axis('off')
ax[1].plot(prof, range(nz)); ax[1].invert_yaxis()
ax[1].axhline(nz // 2, ls='--', c='g', label='centre (d=0)')
ax[1].axhline(peak, ls='-', c='r', label=f'peak layer {peak} (d={peak - nz // 2:+d})')
ax[1].set_xlabel('mean CT'); ax[1].set_ylabel('layer'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'valid {valid.sum() / 1e6:.2f} Mpx = {valid.sum() * (PX_UM * 1e-4) ** 2:.2f} cm2')


## 6. Sweep — both GPUs

`run_depth_sweep.sh` starts its villa and hecate queues in one process, so both see one
`CUDA_VISIBLE_DEVICES` and the second T4 idles. `sweep_dual.py` splits them into two
processes instead: **the three villa models on GPU 0, hecate alone on GPU 1.**

**hecate is dropped by default.** On the control it scored AUC 0.902 against dnative's
0.967, and the three villa models ensemble to **0.979 without it** versus 0.972 with it —
while costing ~14x more per job (58 min vs 4 min). It made a 1.5 h sweep into a 6 h one
for nothing. With hecate gone the two GPUs split the depth range instead of the models.

Heartbeat every 60 s with a per-model breakdown. A flat counter between completions is
normal — maps are written only when a job finishes. Interrupting is safe.


In [ ]:
!{PY} {REPO}/sprint/sweep_dual.py --render {RENDER} --out {MAPS} --dmin {DMIN} --dmax {DMAX} --villa-par {VILLA_PAR} --hec-par {HEC_PAR} --models '{MODELS}'


## 7. Ensemble

One contrast stretch per model fixed across every swept depth, then the four models
averaged as z-scores. That is the step separating this from the single-checkpoint runs
that have surely already been done on these public renders, and it is what the PHerc0343
win turned on.


In [ ]:
!bash {REPO}/sprint/ensemble.sh {RENDER} {MAPS} {IMAGES} '{DMIN}..{DMAX}' '-1 0 1'


## 8. Score

Identical metrics for control and target, so the target reads against the control rather
than against intuition.

- `auc_vs_labels` (control only) is **the** number. **≥ ~0.8**: the pipeline recovers known
  ink, so a null elsewhere is a real statement about that patch. **≈ 0.5–0.6**: our
  adaptation is broken and every result so far is void.
- `reading_vs_blind`: ink sits on one face, so a true positive should be low.
- `row_periodicity`: strongest autocorrelation in the 3–12 mm band, where line spacing
  (5–9 mm) lives.

For reference, PHerc0800 `20251028222030` scored reading_vs_blind **0.13**,
row_periodicity **0.13** at 4.9 mm.


In [ ]:
lab = f'--labels {LABELS}' if LABELS else ''
rot = '--rot180' if ROT180 else ''
!{PY} {REPO}/sprint/score_maps.py --images {IMAGES} --name {NAME} --px-um {PX_UM} {lab} --face {FACE} {rot} --all-stacks --tag {NAME} --json-out {IMAGES}/{NAME}__score.json


## 9. Look

Reading side, blind side and CT at each written depth. **Ink makes strokes of roughly
constant width that join into letterforms; artefacts make soft clouds.** On the control,
a labelled letter is ~2.3 mm (250 px) tall with ~0.6 mm (65 px) strokes — so scale alone
never distinguishes them, only shape does.

On the control the red overlay is the hand-drawn ground truth: compare where the
ensemble fires against where the ink actually is.


In [ ]:
import glob
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

def show(path, overlay=None, title=''):
    im = np.asarray(Image.open(path).convert('L'))
    if ROT180:
        im = im[::-1, ::-1]   # undo the presentation rotation, so it matches the labels
    f = max(im.shape) / 1500.0
    rgb = np.dstack([im] * 3)
    if overlay is not None:
        ink = overlay > 0
        rgb[ink] = (0.45 * rgb[ink] + 0.55 * np.array([255, 60, 60])).astype(np.uint8)
    small = Image.fromarray(rgb).resize((int(im.shape[1] / f), int(im.shape[0] / f)))
    plt.figure(figsize=(15, 15)); plt.imshow(small); plt.axis('off')
    plt.title(title or os.path.basename(path)); plt.show()

gt = np.load(LABELS + '__inklabels.npy') if LABELS else None
read_sfx = '_blind' if FACE == 'blind' else ''
blind_sfx = '' if FACE == 'blind' else '_blind'
for d in ('d+00', 'd+01', 'd-01'):
    for sfx, what in ((read_sfx, 'READING side'), (blind_sfx, 'blind side')):
        p = f'{IMAGES}/{NAME}__ensemble{sfx}__{d}.png'
        if os.path.isfile(p):
            show(p, gt if (what.startswith('READING') and gt is not None) else None,
                 f'{what} {d}' + ('  (red = ground-truth ink)' if gt is not None and what.startswith('READING') else ''))


## 10. Save everything

`/kaggle/temp` is wiped when the session ends. Images, scores, stats and logs are small
and always saved; the raw maps are ~7 MB each and optional, but saving them means the
ensemble can be recomputed later without another GPU hour.

**Commit a notebook version** afterwards, or `/kaggle/working` goes too.


In [ ]:
SAVE_MAPS = True
!mkdir -p {SAVE}
!cp -r {IMAGES}/. {SAVE}/ 2>/dev/null
!mkdir -p {SAVE}/logs && cp -r {MAPS}/logs/. {SAVE}/logs/ 2>/dev/null
if LABELS:
    !cp {LABELS}__inklabels.npy {LABELS}__supervision.npy {LABELS}__crop.json {SAVE}/ 2>/dev/null
if SAVE_MAPS:
    !cd {MAPS} && tar czf {SAVE}/maps_{NAME}.tar.gz *.tif *.png 2>/dev/null

import json, subprocess
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
json.dump({'what': WHAT, 'name': NAME, 'render': RENDER, 'px_um': PX_UM,
           'dmin': DMIN, 'dmax': DMAX, 'ink_det_commit': commit,
           'villa_par': VILLA_PAR, 'hec_par': HEC_PAR},
          open(f'{SAVE}/run.json', 'w'), indent=1)
!du -sh {SAVE}; ls {SAVE}
